<a href="https://colab.research.google.com/github/vladislavpykhteev/HSE_Vladislav_Pykhteev/blob/main/final/final_hw_efrsb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏛️ Итоговое домашнее задание
# «Аналитик реестра — Обработка данных ЕФРСБ»





Вы — юрист, анализирующий данные из **Единого федерального реестра сведений о банкротстве (ЕФРСБ)**.
Вам поступила выгрузка в формате JSON с информацией о сообщениях по делам о банкротстве за 2025 год.

**Ваша задача** — извлечь ключевые данные, очистить от мусора, сопоставить с реестром организаций
и сформировать аналитический отчёт для руководства.

---
### 📋 Как выполнять и сдавать задание

**Где выполнять:**
- **Вариант 1 (рекомендуется):** Загрузите файл `.ipynb` в [Google Colab](https://colab.research.google.com/), загрузите папку `final_hw_data/` в среду выполнения (например, через боковую панель «Файлы» или смонтировав Google Drive).
- **Вариант 2:** Работайте локально в Jupyter Notebook / JupyterLab. Убедитесь, что папка `final_hw_data/` находится в той же директории, что и файл задания.

**Как сдавать на проверку:**
- Отправьте **ссылку на Google Colab** с общим доступом («Любой, у кого есть ссылка — комментатор»), **ИЛИ**
- Отправьте **ссылку на GitHub-репозиторий**, в котором лежит выполненный `.ipynb`-файл и папка `final_hw_data/` с результатами.

> ⚠️ Убедитесь, что ссылка открывается в режиме просмотра и все ячейки выполнены (результаты видны).

---

### 📦 Исходные данные

В папке `final_hw_data/` находятся 3 файла:

| Файл | Описание |
|------|----------|
| `bankruptcy_messages.json` | Сообщения ЕФРСБ |
| `organizations.json` | Реестр организаций |
| `priority_cases.txt` | Номера приоритетных дел |



---# ⚙️ Подготовка: загрузка исходных данных---В Colab при каждом новом запуске нет ни папок, ни файлов. Запусти ячейку ниже — появятся **две кнопки**:- **🌥 Скачать из GitHub** — автоматически тянет три исходных файла из моего публичного репозитория. Запустил и забыл.- **📤 Загрузить с компьютера** — открывает стандартный диалог Colab для выбора файлов на твоём компьютере. Удобно если файлы уже лежат локально или нет интернета.После выбора варианта в этой же ячейке появится отчёт о загруженных файлах. Если файлы уже есть в папке — повторно ничего не качается.

In [1]:
# -*- coding: utf-8 -*-
"""
Подготовка - готовим папку final_hw_data/ с тремя исходными файлами:
  - bankruptcy_messages.json
  - organizations.json
  - priority_cases.txt

Запускаем эту ячейку - появятся две кнопки:
  - "Скачать из GitHub"  -> автоматически тянем файлы из моего репозитория
  - "Загрузить с компьютера" -> откроется стандартный диалог выбора файлов в Colab

После выбора варианта - в этой же ячейке появится отчёт о загруженных файлах.
"""

import os
import ipywidgets as widgets
from IPython.display import display, clear_output

# список файлов, которые нужны для всех заданий
SOURCE_FILES = [
    "bankruptcy_messages.json",
    "organizations.json",
    "priority_cases.txt",
]

# папка для исходников - создаём заранее, чтобы оба варианта писали туда
os.makedirs("final_hw_data", exist_ok=True)


def show_status():
    """
    Показывает - какие файлы лежат в папке final_hw_data/.
    Используется после обоих вариантов загрузки.
    """
    print()
    allOk = True
    for fileName in SOURCE_FILES:
        localPath = "final_hw_data/" + fileName
        if os.path.exists(localPath):
            size = os.path.getsize(localPath)
            print(f"  {localPath:45s} ({size} байт)")
        else:
            print(f"  НЕ НАЙДЕН: {localPath}")
            allOk = False
    print()
    if allOk:
        print("Готово - можно запускать остальные ячейки.")
    else:
        print("Не все файлы на месте - перезапусти ячейку и выбери другой вариант.")


def load_from_github(_=None):
    """
    Вариант 1 - качаем все исходные файлы из публичного репозитория.
    Используем стандартный модуль urllib - ничего ставить не нужно.
    """
    import urllib.request

    # сырые URL файлов в моём репозитории (ветка main)
    REPO_RAW = "https://raw.githubusercontent.com/vladislavpykhteev/HSE_Vladislav_Pykhteev/main/final/final_hw_data"

    with output_area:
        clear_output()
        print("Скачиваю файлы из GitHub...")
        for fileName in SOURCE_FILES:
            localPath = "final_hw_data/" + fileName
            if os.path.exists(localPath):
                print(f"  уже есть: {localPath}")
                continue
            url = REPO_RAW + "/" + fileName
            try:
                urllib.request.urlretrieve(url, localPath)
                print(f"  скачан:   {localPath}")
            except Exception as e:
                # ловим всё подряд, чтобы понятно показать, что пошло не так
                print(f"  ОШИБКА при скачивании {fileName}: {e}")
        show_status()


def load_manual(_=None):
    """
    Вариант 2 - открываем диалог выбора файлов с компьютера.
    Это работает только в Google Colab (там есть google.colab.files).
    """
    with output_area:
        clear_output()
        try:
            from google.colab import files
        except ImportError:
            print("ОШИБКА: ручная загрузка работает только в Google Colab.")
            print("Запусти ноутбук в Colab или выбери первый вариант (GitHub).")
            return

        print("Откроется диалог - выбери все три файла:")
        for name in SOURCE_FILES:
            print("  -", name)
        print()

        uploaded = files.upload()  # тут открывается окно выбора файлов

        # все выбранные файлы перекладываем в папку final_hw_data/
        for fileName in uploaded:
            localPath = "final_hw_data/" + fileName
            with open(localPath, "wb") as f:
                f.write(uploaded[fileName])
            print(f"  сохранён: {localPath}")
        show_status()


# создаём две кнопки и область для вывода результата
btn_github = widgets.Button(
    description="Скачать из GitHub",
    button_style="success",
    icon="cloud-download",
    layout=widgets.Layout(width="220px", height="40px"),
)
btn_manual = widgets.Button(
    description="Загрузить с компьютера",
    button_style="info",
    icon="upload",
    layout=widgets.Layout(width="220px", height="40px"),
)
output_area = widgets.Output()

# привязываем функции к кликам
btn_github.on_click(load_from_github)
btn_manual.on_click(load_manual)

# показываем кнопки рядом друг с другом + область вывода под ними
display(widgets.HBox([btn_github, btn_manual]))
display(output_area)

print("Выбери способ загрузки исходных данных - нажми на одну из кнопок выше.")


Output()

Выбери способ загрузки исходных данных - нажми на одну из кнопок выше.


---
# 🟢 Загрузка и кросс-референс (0-2 балла)
---

### Задание 1.1 — Загрузка данных из файлов

Загрузите данные из трёх файлов:
- `bankruptcy_messages.json` → список `messages`
- `organizations.json` → список `organizations`
- `priority_cases.txt` → список `priority_case_numbers`

Выведите количество загруженных записей из каждого источника.

In [2]:
"""
Задание 1.1 - загрузка данных из трёх файлов
В Colab папку final_hw_data/ нужно загрузить рядом с ноутбуком
JSON открываем через json.load(), txt - построчно через splitlines()
В priority_cases.txt каждая строка - отдельный номер дела, поэтому
обязательно убираем пробелы через strip() и пропускаем пустые строки
"""

import json

# путь к папке с данными - если работаем локально или через монтирование Drive
DATA_DIR = "final_hw_data"

# загружаем сообщения ЕФРСБ
with open(f"{DATA_DIR}/bankruptcy_messages.json", "r", encoding="utf-8") as f:
    messages = json.load(f)

# загружаем реестр организаций
with open(f"{DATA_DIR}/organizations.json", "r", encoding="utf-8") as f:
    organizations = json.load(f)

# загружаем приоритетные дела - txt-файл
with open(f"{DATA_DIR}/priority_cases.txt", "r", encoding="utf-8") as f:
    rawLines = f.read().splitlines()
    # очищаем строки от пробелов и убираем пустые
    priority_case_numbers = [line.strip() for line in rawLines if line.strip()]

print("Загружено сообщений ЕФРСБ:", len(messages))
print("Загружено организаций:    ", len(organizations))
print("Загружено приоритетных дел:", len(priority_case_numbers))

Загружено сообщений ЕФРСБ: 54
Загружено организаций:     30
Загружено приоритетных дел: 10


### Задание 1.2 — Словарь организаций

Создайте словарь `org_by_inn`, где ключ — ИНН (строка), а значение — словарь с данными об организации.
Используйте **dict comprehension**.

Выведите количество организаций в словаре и информацию по ИНН `"7701234567"`.

In [3]:
"""
Задание 1.2 - словарь организаций по ИНН
Используем dict comprehension - удобный способ собрать словарь в одну строку
Ключ - ИНН (строка), значение - вся карточка организации
"""

org_by_inn = {org["inn"]: org for org in organizations}

print()
print("Размер словаря org_by_inn:", len(org_by_inn))
print("Пример - организация с ИНН 7701234567:")
print(org_by_inn.get("7701234567"))


Размер словаря org_by_inn: 30
Пример - организация с ИНН 7701234567:
{'inn': '7701234567', 'ogrn': '1027700123456', 'name': 'ООО "Альфа-Строй"', 'address': 'г. Москва, ул. Строителей, д. 15', 'region': 'Москва'}


### Задание 1.3 — Кросс-референс: связываем сообщения с организациями

Для каждого сообщения добавьте поле `org_name` (название организации) и `region`,
используя словарь `org_by_inn` и метод `.get()`.
Если организация не найдена — ставьте значение `None`.

Посчитайте, сколько сообщений удалось связать с организацией,
а сколько — не удалось.

In [4]:
"""
Задание 1.3 - кросс-референс между сообщениями и организациями
Через .get() безопасно получаем словарь по ИНН - если его нет, вернётся None
Тогда в org_name и region тоже пишем None
По ходу считаем сколько удалось/не удалось связать
"""

linkedCount = 0
notLinkedCount = 0

for msg in messages:
    inn = msg.get("publisher_inn")
    org = org_by_inn.get(inn)

    if org is not None:
        msg["org_name"] = org.get("name")
        msg["region"] = org.get("region")
        linkedCount = linkedCount + 1
    else:
        msg["org_name"] = None
        msg["region"] = None
        notLinkedCount = notLinkedCount + 1

print()
print("Связано с организацией:    ", linkedCount)
print("Не удалось связать:        ", notLinkedCount)


Связано с организацией:     51
Не удалось связать:         3


### Задание 1.4 — Множества и приоритетные дела

1. Создайте множество `priority_set` из списка `priority_case_numbers`.
2. Создайте множество `message_case_set` из номеров дел в `messages`
   (используйте list comprehension + filter для непустых номеров).
3. Найдите пересечение — номера дел, которые есть и в приоритетных, и в сообщениях (`&`).
4. Выведите результат.

In [5]:
"""
Задание 1.4 - множества и приоритетные дела
1) priority_set - множество из списка приоритетных дел
2) message_case_set - множество номеров дел из сообщений (только непустые)
3) пересечение через оператор &
"""

# множество приоритетных дел
priority_set = set(priority_case_numbers)

# множество номеров дел из сообщений - list comprehension с фильтрацией пустых
message_case_set = set(
    [msg["case_number"] for msg in messages if msg.get("case_number")]
)

# пересечение - дела, которые есть и там и там
intersection = priority_set & message_case_set

print()
print("Приоритетных дел всего:        ", len(priority_set))
print("Уникальных номеров в сообщениях:", len(message_case_set))
print("Пересечение (приоритет и есть в сообщениях):", len(intersection))
print("Список общих номеров:")
for caseNumber in sorted(intersection):
    print("  -", caseNumber)


Приоритетных дел всего:         10
Уникальных номеров в сообщениях: 16
Пересечение (приоритет и есть в сообщениях): 10
Список общих номеров:
  - А60-11111/2025
  - А60-12345/2025
  - А60-22222/2025
  - А60-33333/2025
  - А60-44444/2025
  - А60-56789/2025
  - А60-66666/2025
  - А60-77777/2025
  - А60-88888/2025
  - А60-99999/2025


---
# 🟡 Очистка и валидация (0-3 балла)
---

### Задание 2.1 — Функция парсинга дат

Напишите функцию `parse_date(date_str)`, которая принимает строку с датой
и возвращает объект `datetime` или `None`, если распарсить не удалось.

Форматы для обработки:
- `"DD.MM.YYYY"` — `strptime`
- `"YYYY-MM-DD"` — `fromisoformat`
- `"DD месяца YYYY г."` — замена русских месяцев + `strptime`
- `"DD/MM/YYYY HH:MM"` — `strptime`

Обрабатывайте `None` и пустые строки через `try/except`.

In [6]:
"""
Задание 2.1 - функция парсинга дат
В данных встречаются 4 формата:
  - "DD.MM.YYYY"          - русский, парсим через strptime
  - "YYYY-MM-DD"          - ISO, удобно через datetime.fromisoformat()
  - "DD месяца YYYY г."   - русское написание месяца, нужно заменить на номер
  - "DD/MM/YYYY HH:MM"    - со временем, тоже через strptime

Если на вход пришёл None, пустая строка или строка вида "ДАТА УТЕРЯНА"
нужно вернуть None и не падать с ошибкой - оборачиваем в try/except

Идею с перебором форматов через список я подсмотрел в примере преподавателя
legaltech_datetime - это удобнее, чем 4 отдельных if-блока
"""

from datetime import datetime

# словарь для замены русских месяцев на номера
# нужен для дат типа "15 марта 2025 г."
RUSSIAN_MONTHS = {
    "января": "01", "февраля": "02", "марта": "03", "апреля": "04",
    "мая": "05", "июня": "06", "июля": "07", "августа": "08",
    "сентября": "09", "октября": "10", "ноября": "11", "декабря": "12",
}

# список шаблонов strptime, которые перебираем по очереди
# порядок важен - сначала более частые форматы
DATE_FORMATS = [
    "%d.%m.%Y",         # 15.01.2025
    "%d/%m/%Y %H:%M",   # 03/04/2025 14:30
    "%Y-%m-%d",         # 2025-02-10 (на случай если fromisoformat не сработает)
    "%d %m %Y",         # после замены русского месяца на номер
]


def parse_date(date_str):
    """
    Парсит строку с датой. Возвращает datetime или None при ошибке.
    Не падает - все ValueError ловим через try/except.
    """
    # сначала отсекаем None и пустую строку
    if date_str is None or not isinstance(date_str, str):
        return None

    s = date_str.strip()
    if s == "":
        return None

    # сперва пробуем ISO 8601 - самый быстрый путь, не нужен шаблон
    # пример из вебинара: datetime.fromisoformat('2024-01-20T14:30:00+02:00')
    try:
        return datetime.fromisoformat(s)
    except ValueError:
        pass

    # если в строке есть русское название месяца - заменяем его на номер
    # это нужно сделать ДО strptime, чтобы шаблон "%d %m %Y" сработал
    sNormalized = s.lower().replace(" г.", "").replace(" г", "")
    for ruMonth, numMonth in RUSSIAN_MONTHS.items():
        if ruMonth in sNormalized:
            sNormalized = sNormalized.replace(ruMonth, numMonth)
            break

    # перебираем шаблоны по очереди - какой подойдёт, тот и используем
    for fmt in DATE_FORMATS:
        try:
            # пробуем сначала исходную строку, потом нормализованную
            return datetime.strptime(s, fmt)
        except ValueError:
            try:
                return datetime.strptime(sNormalized, fmt)
            except ValueError:
                continue

    # ни один формат не подошёл - вернём None
    # вызывающий код решит, что с этим делать (записать в ошибки или пропустить)
    return None


# проверяем функцию на разных вариантах
print()
print("Проверяем parse_date():")
print("  '15.01.2025'        ->", parse_date("15.01.2025"))
print("  '2025-02-10'        ->", parse_date("2025-02-10"))
print("  '03/04/2025 14:30'  ->", parse_date("03/04/2025 14:30"))
print("  '15 марта 2025 г.'  ->", parse_date("15 марта 2025 г."))
print("  '' (пустая строка)  ->", parse_date(""))
print("  None                ->", parse_date(None))
print("  'ДАТА УТЕРЯНА'      ->", parse_date("ДАТА УТЕРЯНА"))


Проверяем parse_date():
  '15.01.2025'        -> 2025-01-15 00:00:00
  '2025-02-10'        -> 2025-02-10 00:00:00
  '03/04/2025 14:30'  -> 2025-04-03 14:30:00
  '15 марта 2025 г.'  -> 2025-03-15 00:00:00
  '' (пустая строка)  -> None
  None                -> None
  'ДАТА УТЕРЯНА'      -> None


### Задание 2.2 — Функция валидации сообщения

Напишите функцию `validate_message(msg)`, которая:

1. Проверяет наличие **обязательных полей**: `publisher_inn`, `msg_text`, `date_published`, `type`, `case_number`.
   Если поле отсутствует — ошибка типа `KeyError`.
2. Проверяет, что **ИНН** состоит из 10 или 12 цифр (метод `.isdigit()`).
3. Парсит дату через `parse_date()`. Если дата не парсится — ошибка.
4. Проверяет, что **тип сообщения** — непустая строка.

Функция возвращает кортеж `(valid_msg, errors)`:
- `valid_msg` — словарь с очищенными данными (или `None`)
- `errors` — список строк с описанием ошибок

In [7]:
"""
Задание 2.2 - функция валидации сообщения
Проверяем:
  1) наличие обязательных полей - если нет, бросаем KeyError
  2) ИНН - 10 или 12 цифр, проверка через .isdigit()
  3) дата - парсится через parse_date()
  4) тип сообщения - непустая строка
Если что-то не так - складываем описание ошибки в errors
В valid_msg возвращаем словарь с очищенными значениями (либо None если ошибки)
"""

REQUIRED_FIELDS = ["publisher_inn", "msg_text", "date_published", "type", "case_number"]


def validate_message(msg):
    errors = []
    cleaned = {}

    # 1) проверка обязательных полей
    for field in REQUIRED_FIELDS:
        if field not in msg:
            # фиксируем ошибку, но не прерываемся - чтобы собрать все ошибки сразу
            errors.append(f"KeyError: отсутствует поле '{field}'")

    # если хотя бы одно поле отсутствует - дальше проверять смысла нет
    if errors:
        return None, errors

    # 2) проверка ИНН - 10 или 12 цифр
    inn = msg.get("publisher_inn")
    if inn is None or not str(inn).isdigit():
        errors.append("ValueError: ИНН должен состоять только из цифр")
    elif len(str(inn)) not in (10, 12):
        errors.append(f"ValueError: ИНН должен быть длиной 10 или 12 цифр, получено {len(str(inn))}")

    # 3) проверка даты - парсим через нашу функцию
    parsedDate = parse_date(msg.get("date_published"))
    if parsedDate is None:
        errors.append(f"ValueError: не удалось распарсить дату '{msg.get('date_published')}'")

    # 4) тип сообщения - не None и не пустая строка
    msgType = msg.get("type")
    if msgType is None or not isinstance(msgType, str) or msgType.strip() == "":
        errors.append("ValueError: тип сообщения должен быть непустой строкой")

    # дополнительно проверим номер дела - чтобы не было пустого
    caseNumber = msg.get("case_number")
    if caseNumber is None or str(caseNumber).strip() == "":
        errors.append("ValueError: номер дела должен быть непустой строкой")

    # если были ошибки - возвращаем None
    if errors:
        return None, errors

    # собираем "чистую" копию сообщения
    cleaned = {
        "publisher_inn":  str(inn),
        "msg_text":       msg.get("msg_text", ""),
        "date_published": parsedDate,
        "type":           msgType.strip(),
        "case_number":    str(caseNumber).strip(),
        # не забываем доп. поля из задания 1.3
        "org_name":       msg.get("org_name"),
        "region":         msg.get("region"),
    }
    return cleaned, []


# проверяем функцию на одном сообщении
print()
print("Проверяем validate_message() на первом сообщении:")
sampleValid, sampleErr = validate_message(messages[0])
print("  errors :", sampleErr)
print("  msg    :", sampleValid)


Проверяем validate_message() на первом сообщении:
  errors : []
  msg    : {'publisher_inn': '7701234567', 'msg_text': 'Сообщение о введении процедуры наблюдения в отношении ООО "Альфа-Строй". Арбитражный управляющий Иванов Иван Иванович. Долг составляет 15 000 000 руб. АС Свердловской области.', 'date_published': datetime.datetime(2025, 1, 15, 0, 0), 'type': 'Введение процедуры', 'case_number': 'А60-12345/2025', 'org_name': 'ООО "Альфа-Строй"', 'region': 'Москва'}


### Задание 2.3 — Массовая валидация

Примените `validate_message()` ко всем сообщениям.
Разделите результат на два списка: `valid_messages` и `error_records`.

Соберите **статистику ошибок** по типам (сколько `KeyError`, `ValueError` и т.д.).

In [8]:
"""
Задание 2.3 - массовая валидация
Прогоняем все сообщения через validate_message()
Делим результат на два списка - валидные и с ошибками
По ходу собираем статистику по типам ошибок (KeyError / ValueError)

Из примера преподавателя legaltech_error_handling взял идею с функцией classify_error -
она разделяет ошибки по типам (ValueError / KeyError) для отчёта
"""

valid_messages = []
error_records = []
errorStats = {}

for msg in messages:
    cleaned, errs = validate_message(msg)
    if cleaned is not None:
        valid_messages.append(cleaned)
    else:
        # сохраняем исходное сообщение и список ошибок
        error_records.append({
            "original": msg,
            "errors": errs,
        })
        # подсчитываем типы ошибок (часть до двоеточия)
        for e in errs:
            errType = e.split(":")[0]
            errorStats[errType] = errorStats.get(errType, 0) + 1

print()
print("Всего сообщений:        ", len(messages))
print("Валидных сообщений:     ", len(valid_messages))
print("С ошибками:             ", len(error_records))
print("Статистика по типам ошибок:")
for errType, count in errorStats.items():
    print(f"  {errType}: {count}")


Всего сообщений:         54
Валидных сообщений:      48
С ошибками:              6
Статистика по типам ошибок:
  ValueError: 6


---
# 🔵 Извлечение данных и аналитика (0-2 балла)
---

### Задание 3.1 — Извлечение сумм из текста

Напишите функцию `extract_amounts(text)`, которая ищет упоминания денежных сумм в тексте сообщения.

Используйте строковые методы.

Ищите по ключевым словам: `"руб."`, `"тыс. руб."`, `"млн руб."`.

Функция возвращает список строк — найденных сумм.

In [9]:
"""
Задание 3.1 - функция извлечения сумм из текста
Идём по тексту словами через .split() и ищем ключевые слова из задания:
"руб.", "тыс. руб." и "млн руб."
Собираем кусочки слева от ключевого слова - это и будет сумма с обозначением
Возвращаем список найденных строк
"""

# ключевые слова - порядок важен, сначала самые длинные, чтобы не путать
KEYWORDS = ["млн руб.", "тыс. руб.", "руб."]


def extract_amounts(text):
    if not text or not isinstance(text, str):
        return []

    results = []
    # для удобства поиска приведём к одному регистру
    workText = text

    for keyword in KEYWORDS:
        # ищем все вхождения ключевого слова
        startPos = 0
        while True:
            foundPos = workText.find(keyword, startPos)
            if foundPos == -1:
                break

            # берём 30 символов слева - этого хватит для большинства сумм
            leftStart = max(0, foundPos - 30)
            leftPart = workText[leftStart:foundPos]

            # из левой части берём последние "слова" - цифры и пробелы
            # двигаемся справа налево пока встречаем цифры, пробелы или точки
            collected = ""
            for ch in reversed(leftPart):
                if ch.isdigit() or ch in " .,":
                    collected = ch + collected
                else:
                    break

            amountStr = collected.strip() + " " + keyword
            amountStr = amountStr.strip()
            if amountStr and amountStr != keyword:
                results.append(amountStr)

            # сдвигаем стартовую позицию, чтобы найти следующее вхождение
            startPos = foundPos + len(keyword)

            # подменяем найденный фрагмент пробелами в workText, чтобы более
            # короткое ключевое слово ("руб.") не нашло его повторно
            workText = workText[:foundPos] + (" " * len(keyword)) + workText[foundPos + len(keyword):]

    return results


# проверим функцию
print()
print("Проверяем extract_amounts():")
testText1 = "Долг составляет 15 000 000 руб. Также есть требования на 8 500 тыс. руб. и 2 млн руб."
print("  Текст:", testText1)
print("  Найдено:", extract_amounts(testText1))


Проверяем extract_amounts():
  Текст: Долг составляет 15 000 000 руб. Также есть требования на 8 500 тыс. руб. и 2 млн руб.
  Найдено: ['2 млн руб.', '8 500 тыс. руб.', '15 000 000 руб.']


### Задание 3.2 — Поиск упоминаний арбитражных судов

Напишите функцию `find_court_mentions(text)`, которая проверяет,
содержит ли текст упоминание арбитражного суда.

Ищите подстроку `"АС "` (с пробелом) в тексте через оператор `in`.
Верните `True` / `False`.

In [10]:
"""
Задание 3.2 - проверка упоминаний арбитражных судов
Простая функция - проверяем подстроку "АС " в тексте через оператор in
Возвращаем True или False
"""

def find_court_mentions(text):
    if not text or not isinstance(text, str):
        return False
    return "АС " in text


# проверим
print()
print("Проверяем find_court_mentions():")
print("  'АС Свердловской области'    ->", find_court_mentions("АС Свердловской области"))
print("  'обычный текст без суда'     ->", find_court_mentions("обычный текст без суда"))


Проверяем find_court_mentions():
  'АС Свердловской области'    -> True
  'обычный текст без суда'     -> False


### Задание 3.3 — Извлечение ФИО арбитражных управляющих

Напишите функцию `extract_manager_name(text)`, которая ищет ФИО арбитражного управляющего.

Алгоритм:
1. Проверьте, содержит ли текст подстроку `"арбитражный управляющий"`.
2. Если да — найдите позицию этой подстроки, возьмите текст после неё.
3. С помощью `.split()` извлеките следующие 3 слова (ФИО).
4. Соедините через пробел и верните.
5. Если не найдено — верните `None`.

In [11]:
"""
Задание 3.3 - извлечение ФИО арбитражного управляющего
Алгоритм по шагам:
1) ищем подстроку "арбитражный управляющий" (через .lower() для надёжности)
2) если нашли - берём текст после неё
3) разбиваем по пробелам через split() и берём первые 3 "слова"
4) объединяем через пробел
"""

KEYWORD_AM = "арбитражный управляющий"


def extract_manager_name(text):
    if not text or not isinstance(text, str):
        return None

    lowerText = text.lower()
    pos = lowerText.find(KEYWORD_AM)
    if pos == -1:
        return None

    # берём текст после ключевого слова, но из исходной строки
    # чтобы регистр ФИО сохранился
    afterText = text[pos + len(KEYWORD_AM):]

    # разбиваем по пробелам и берём первые три слова
    words = afterText.split()
    if len(words) < 3:
        return None

    fio = words[0] + " " + words[1] + " " + words[2]

    # уберём знаки препинания на конце слов (например "Иванович.")
    fioClean = fio.strip(" .,;:")
    return fioClean


# проверим
print()
print("Проверяем extract_manager_name():")
testText3 = "Назначен арбитражный управляющий Иванов Иван Иванович, далее..."
print("  Текст:", testText3)
print("  ФИО:  ", extract_manager_name(testText3))


Проверяем extract_manager_name():
  Текст: Назначен арбитражный управляющий Иванов Иван Иванович, далее...
  ФИО:   Иванов Иван Иванович


### Задание 3.4 — Обогащение данных

Для каждого **валидного** сообщения добавьте поля:
- `amounts` — список извлечённых сумм (функция `extract_amounts`)
- `has_court_mention` — `True`/`False` (функция `find_court_mentions`)
- `manager_name` — ФИО или `None` (функция `extract_manager_name`)

In [12]:
"""
Задание 3.4 - обогащение данных
Для каждого валидного сообщения добавляем три поля:
  amounts          - список сумм из текста
  has_court_mention - True/False
  manager_name     - ФИО или None
"""

for msg in valid_messages:
    text = msg.get("msg_text", "")
    msg["amounts"] = extract_amounts(text)
    msg["has_court_mention"] = find_court_mentions(text)
    msg["manager_name"] = extract_manager_name(text)

# покажем 3 первых сообщения для проверки
print()
print("Пример обогащённых сообщений (первые 3):")
for m in valid_messages[:3]:
    print("  -", m["case_number"])
    print("    суммы:", m["amounts"])
    print("    суд:  ", m["has_court_mention"])
    print("    ФИО:  ", m["manager_name"])


Пример обогащённых сообщений (первые 3):
  - А60-12345/2025
    суммы: ['15 000 000 руб.']
    суд:   True
    ФИО:   Иванов Иван Иванович
  - А60-56789/2025
    суммы: ['8 500 тыс. руб.']
    суд:   True
    ФИО:   None
  - А60-11111/2025
    суммы: ['42 млн руб.']
    суд:   False
    ФИО:   Петров Пётр Петрович


### Задание 3.5 — Аналитика

Постройте следующие показатели по **валидным** сообщениям:

1. **Количество сообщений по типам** — словарь `{тип: количество}`
2. **Количество сообщений по регионам** — словарь `{регион: количество}`, пропуская `None`
3. **Топ-5 публикаторов** по количеству сообщений — `sorted(key=lambda...)`
4. **Таймлайн** — сообщения, отсортированные по дате публикации

In [13]:
"""
Задание 3.5 - аналитика по валидным сообщениям
1) количество сообщений по типам
2) количество сообщений по регионам (None пропускаем)
3) топ-5 публикаторов по количеству сообщений (по ИНН)
4) таймлайн - сообщения по дате публикации
"""

# 1) по типам
type_stats = {}
for m in valid_messages:
    t = m["type"]
    type_stats[t] = type_stats.get(t, 0) + 1

# 2) по регионам - пропускаем None
region_stats = {}
for m in valid_messages:
    region = m.get("region")
    if region is None:
        continue
    region_stats[region] = region_stats.get(region, 0) + 1

# 3) топ-5 публикаторов - сначала собираем словарь {ИНН: счётчик}
publisherStats = {}
for m in valid_messages:
    inn = m["publisher_inn"]
    publisherStats[inn] = publisherStats.get(inn, 0) + 1

# сортируем по убыванию через sorted с lambda
top_publishers = sorted(
    publisherStats.items(),
    key=lambda pair: pair[1],
    reverse=True,
)[:5]

# 4) таймлайн - сортируем по date_published (там у нас уже datetime)
timeline = sorted(valid_messages, key=lambda m: m["date_published"])

print()
print("Статистика по типам сообщений:")
for t, c in type_stats.items():
    print(f"  {t}: {c}")

print()
print("Статистика по регионам:")
for r, c in region_stats.items():
    print(f"  {r}: {c}")

print()
print("Топ-5 публикаторов:")
for inn, count in top_publishers:
    orgName = org_by_inn.get(inn, {}).get("name", "—")
    print(f"  ИНН {inn} ({orgName}): {count} сообщений")

print()
print("Таймлайн (первые 5 сообщений по дате):")
for m in timeline[:5]:
    dateStr = m["date_published"].strftime("%d.%m.%Y")
    print(f"  {dateStr} | {m['type']} | {m['case_number']}")


Статистика по типам сообщений:
  Введение процедуры: 8
  Собрание кредиторов: 3
  Завершение процедуры: 6
  Признание банкротом: 3
  Продажа имущества: 6
  Требование кредитора: 4
  Оспаривание сделки: 3
  Подача заявления: 3
  Отстранение управляющего: 1
  Мировое соглашение: 3
  Субсидиарная ответственность: 2
  Передача дела: 3
  Назначение управляющего: 2
  Жалоба на управляющего: 1

Статистика по регионам:
  Москва: 25
  Свердловская область: 6
  Санкт-Петербург: 5
  Краснодарский край: 3
  Челябинская область: 3
  Ростовская область: 2
  Владимирская область: 1
  Московская область: 2

Топ-5 публикаторов:
  ИНН 7701234567 (ООО "Альфа-Строй"): 3 сообщений
  ИНН 7706567890 (ООО "Тета-Консалтинг"): 3 сообщений
  ИНН 7702345678 (ЗАО "Бета-Инвест"): 2 сообщений
  ИНН 6658123450 (ООО "Гамма-Трейд"): 2 сообщений
  ИНН 7810345612 (ОАО "Дельта-Логистик"): 2 сообщений

Таймлайн (первые 5 сообщений по дате):
  10.01.2025 | Завершение процедуры | А60-25814/2025
  15.01.2025 | Введение проце

---
# 🟣 Отчётность (0-1 балл)
---

### Задание 4.1 — Подготовка данных для сохранения

Подготовьте данные для записи в JSON. Даты нужно преобразовать обратно в строки (`strftime`),
чтобы JSON был читаемым.

In [14]:
"""
Задание 4.1 - подготовка данных для записи в JSON
JSON не умеет сериализовать datetime, поэтому преобразуем дату обратно в строку
через strftime() в формате DD.MM.YYYY
Делаем копию каждого словаря, чтобы не портить исходные данные
"""

valid_messages_for_json = []
for m in valid_messages:
    copyMsg = dict(m)  # делаем копию словаря
    if isinstance(copyMsg.get("date_published"), datetime):
        copyMsg["date_published"] = copyMsg["date_published"].strftime("%d.%m.%Y")
    valid_messages_for_json.append(copyMsg)

print()
print("Подготовлено валидных сообщений для JSON:", len(valid_messages_for_json))
print("Пример:", valid_messages_for_json[0])


Подготовлено валидных сообщений для JSON: 48
Пример: {'publisher_inn': '7701234567', 'msg_text': 'Сообщение о введении процедуры наблюдения в отношении ООО "Альфа-Строй". Арбитражный управляющий Иванов Иван Иванович. Долг составляет 15 000 000 руб. АС Свердловской области.', 'date_published': '15.01.2025', 'type': 'Введение процедуры', 'case_number': 'А60-12345/2025', 'org_name': 'ООО "Альфа-Строй"', 'region': 'Москва', 'amounts': ['15 000 000 руб.'], 'has_court_mention': True, 'manager_name': 'Иванов Иван Иванович'}


### Задание 4.2 — Сохранение `analysis_results.json`

Сохраните файл `analysis_results.json` со следующей структурой:
```json
{
  "valid_messages": [...],
  "type_stats": {...},
  "region_stats": {...},
  "priority_messages": [...]
}
```

In [15]:
"""
Задание 4.2 - сохранение analysis_results.json
Пишем итоговую структуру: валидные сообщения, статистики и приоритетные дела
ensure_ascii=False - чтобы русский текст в файле сохранялся читаемым
indent=2 - чтобы файл был удобен для чтения человеком
"""

# приоритетные сообщения - те, чьё дело попало в priority_set
priority_messages = [m for m in valid_messages_for_json if m["case_number"] in priority_set]

analysisResult = {
    "valid_messages":    valid_messages_for_json,
    "type_stats":        type_stats,
    "region_stats":      region_stats,
    "priority_messages": priority_messages,
}

with open(f"{DATA_DIR}/analysis_results.json", "w", encoding="utf-8") as f:
    json.dump(analysisResult, f, ensure_ascii=False, indent=2)

print()
print("Файл analysis_results.json сохранён")
print("  всего валидных:    ", len(valid_messages_for_json))
print("  приоритетных:      ", len(priority_messages))


Файл analysis_results.json сохранён
  всего валидных:     48
  приоритетных:       32


### Задание 4.3 — Сохранение `validation_errors.json`

Сохраните проблемные записи с описанием ошибок.

In [16]:
"""
Задание 4.3 - сохранение validation_errors.json
В error_records у нас уже сохранены проблемные записи и описания ошибок
"""

with open(f"{DATA_DIR}/validation_errors.json", "w", encoding="utf-8") as f:
    json.dump(error_records, f, ensure_ascii=False, indent=2)

print()
print("Файл validation_errors.json сохранён, записей:", len(error_records))


Файл validation_errors.json сохранён, записей: 6


### Задание 4.4 — Текстовый отчёт `summary_report.txt`

Сформируйте текстовый аналитический отчёт для руководства с помощью **f-строк**.

Отчёт должен содержать:
- Заголовок и дату
- Общую статистику (всего сообщений, валидных, ошибочных)
- Статистику по типам сообщений
- Статистику по регионам
- Топ-5 публикаторов
- Список приоритетных дел
- Список найденных арбитражных управляющих

In [17]:
"""
Задание 4.4 - текстовый отчёт summary_report.txt
Собираем строку через f-строки, потом записываем в файл
В отчёт включаем заголовок, дату, общую статистику, по типам, по регионам,
топ-5 публикаторов, приоритетные дела и список арбитражных управляющих
"""

# дата формирования отчёта - берём текущую
reportDate = datetime.now().strftime("%d.%m.%Y %H:%M")

# собираем уникальные ФИО арбитражных управляющих
managers = set()
for m in valid_messages:
    if m.get("manager_name"):
        managers.add(m["manager_name"])

# составляем отчёт построчно через список и потом склеиваем через "\n"
lines = []
lines.append("=" * 60)
lines.append("АНАЛИТИЧЕСКИЙ ОТЧЁТ ПО ДАННЫМ ЕФРСБ")
lines.append(f"Дата формирования: {reportDate}")
lines.append("=" * 60)
lines.append("")

lines.append("1. ОБЩАЯ СТАТИСТИКА")
lines.append("-" * 60)
lines.append(f"Всего сообщений на входе:   {len(messages)}")
lines.append(f"Валидных сообщений:         {len(valid_messages)}")
lines.append(f"Сообщений с ошибками:       {len(error_records)}")
lines.append(f"Связано с организацией:     {linkedCount}")
lines.append(f"Не связано с организацией:  {notLinkedCount}")
lines.append("")

lines.append("2. СТАТИСТИКА ПО ТИПАМ СООБЩЕНИЙ")
lines.append("-" * 60)
for t, c in sorted(type_stats.items(), key=lambda x: x[1], reverse=True):
    lines.append(f"  {t}: {c}")
lines.append("")

lines.append("3. СТАТИСТИКА ПО РЕГИОНАМ")
lines.append("-" * 60)
for r, c in sorted(region_stats.items(), key=lambda x: x[1], reverse=True):
    lines.append(f"  {r}: {c}")
lines.append("")

lines.append("4. ТОП-5 ПУБЛИКАТОРОВ")
lines.append("-" * 60)
for i, (inn, count) in enumerate(top_publishers, start=1):
    orgName = org_by_inn.get(inn, {}).get("name", "—")
    lines.append(f"  {i}. ИНН {inn} | {orgName} | сообщений: {count}")
lines.append("")

lines.append("5. ПРИОРИТЕТНЫЕ ДЕЛА (есть в реестре приоритета и в сообщениях)")
lines.append("-" * 60)
if intersection:
    for caseNumber in sorted(intersection):
        # найдём первое сообщение с этим делом для красивого вывода
        firstMsg = next((m for m in valid_messages if m["case_number"] == caseNumber), None)
        if firstMsg:
            lines.append(f"  - {caseNumber} | {firstMsg['type']} | {firstMsg.get('org_name') or '—'}")
        else:
            lines.append(f"  - {caseNumber}")
else:
    lines.append("  (нет совпадений)")
lines.append("")

lines.append("6. АРБИТРАЖНЫЕ УПРАВЛЯЮЩИЕ (упомянуты в сообщениях)")
lines.append("-" * 60)
if managers:
    for fio in sorted(managers):
        lines.append(f"  - {fio}")
else:
    lines.append("  (не найдено)")
lines.append("")
lines.append("=" * 60)
lines.append("Конец отчёта")
lines.append("=" * 60)

reportText = "\n".join(lines)

with open(f"{DATA_DIR}/summary_report.txt", "w", encoding="utf-8") as f:
    f.write(reportText)

print()
print("Файл summary_report.txt сохранён")
print()
print("Превью отчёта (первые 25 строк):")
for line in lines[:25]:
    print(line)


Файл summary_report.txt сохранён

Превью отчёта (первые 25 строк):
АНАЛИТИЧЕСКИЙ ОТЧЁТ ПО ДАННЫМ ЕФРСБ
Дата формирования: 26.04.2026 09:05

1. ОБЩАЯ СТАТИСТИКА
------------------------------------------------------------
Всего сообщений на входе:   54
Валидных сообщений:         48
Сообщений с ошибками:       6
Связано с организацией:     51
Не связано с организацией:  3

2. СТАТИСТИКА ПО ТИПАМ СООБЩЕНИЙ
------------------------------------------------------------
  Введение процедуры: 8
  Завершение процедуры: 6
  Продажа имущества: 6
  Требование кредитора: 4
  Собрание кредиторов: 3
  Признание банкротом: 3
  Оспаривание сделки: 3
  Подача заявления: 3
  Мировое соглашение: 3
  Передача дела: 3


---
## ✅ Итоги

Если вы корректно выполнили все 4 уровня, у вас должны получиться:

| Файл | Описание |
|------|----------|
| `analysis_results.json` | Валидные сообщения + статистика + приоритетные дела |
| `validation_errors.json` | Проблемные записи с описанием ошибок |
| `summary_report.txt` | Текстовый аналитический отчёт для руководства |



---## 🎁 Бонус: автоматический коммит результатов в GitHubЧтобы файлы-результаты (`analysis_results.json`, `validation_errors.json`, `summary_report.txt`) и сам ноутбук попадали в репозиторий **в один клик**, я написал интерактивную ячейку с кнопкой.**Как пользоваться (с секретами Colab — самый безопасный способ):**1. Создай Personal Access Token: [github.com/settings/personal-access-tokens/new](https://github.com/settings/personal-access-tokens/new)   - Тип: **Fine-grained**   - Repository access: только `HSE_Vladislav_Pykhteev`   - Permissions → Contents: **Read and write**2. В Colab открой панель «Секреты» (🔑 слева) → **Добавить секрет**   - Название: `GH_TOKEN`   - Значение: вставь токен   - Включи переключатель **«Доступ из блокнотов»**3. Запусти ячейку ниже и нажми кнопку **«Закоммитить в GitHub»**Если запускаешь не в Colab или предпочитаешь ручной ввод — поменяй имя секрета или просто введи токен в поле как обычный текст.

In [18]:
# -*- coding: utf-8 -*-
"""
Бонусный код - коммит результатов в GitHub-репозиторий через API.
Читаем токен из Секретов Colab (через google.colab.userdata) - так безопаснее,
чем вставлять токен в поле каждый раз.

Что коммитим:
  - сам ноутбук (final_hw_efrsb.ipynb) - ищем его по нескольким путям
  - три файла-результата из final_hw_data/
"""

import os
import glob
import json
import base64
import urllib.request
import urllib.error

import ipywidgets as widgets
from IPython.display import display, clear_output


# настройки репозитория
OWNER = "vladislavpykhteev"
REPO = "HSE_Vladislav_Pykhteev"
BRANCH = "main"

# что коммитим из final_hw_data/
DATA_FILES = {
    "final_hw_data/analysis_results.json":   "final/final_hw_data/analysis_results.json",
    "final_hw_data/validation_errors.json":  "final/final_hw_data/validation_errors.json",
    "final_hw_data/summary_report.txt":      "final/final_hw_data/summary_report.txt",
}

COMMIT_MESSAGE = "Итоговое ДЗ ЕФРСБ - решение и результаты"


def find_notebook_path():
    """
    В Colab ноутбук может лежать в разных местах:
      - /content/final_hw_efrsb.ipynb (если открывали из локальной копии)
      - /content/drive/MyDrive/.../final_hw_efrsb.ipynb (если из Drive)
      - просто final_hw_efrsb.ipynb относительно текущей папки
    Ищем где он есть на самом деле.
    """
    candidates = [
        "final_hw_efrsb.ipynb",
        "/content/final_hw_efrsb.ipynb",
    ]
    # ищем также в подмонтированном Drive (если есть)
    candidates.extend(glob.glob("/content/drive/MyDrive/**/final_hw_efrsb*.ipynb", recursive=True))
    candidates.extend(glob.glob("/content/**/final_hw_efrsb*.ipynb", recursive=True))

    # возвращаем первый найденный путь
    for path in candidates:
        if os.path.exists(path):
            return path
    return None


def github_api(token, method, path, payload=None):
    """
    Универсальная обёртка над GitHub API.
    Возвращает кортеж (статус, тело_ответа_как_словарь_или_строка).
    """
    url = "https://api.github.com" + path
    headers = {
        "Authorization": f"token {token}",
        "Accept": "application/vnd.github+json",
        "User-Agent": "hse-final-hw-script",
    }

    data = None
    if payload is not None:
        data = json.dumps(payload).encode("utf-8")
        headers["Content-Type"] = "application/json"

    request = urllib.request.Request(url, data=data, headers=headers, method=method)
    try:
        with urllib.request.urlopen(request) as response:
            body = response.read().decode("utf-8")
            return response.status, json.loads(body) if body else {}
    except urllib.error.HTTPError as exc:
        body = exc.read().decode("utf-8", errors="replace")
        # пытаемся распарсить ответ как JSON, иначе вернём сырое тело
        try:
            return exc.code, json.loads(body)
        except Exception:
            return exc.code, {"message": body}


def get_existing_sha(token, repo_path):
    """
    Запрашивает SHA текущей версии файла в репо (нужен для апдейта).
    Если файла нет (404) - вернёт None.
    """
    status, response = github_api(
        token, "GET", f"/repos/{OWNER}/{REPO}/contents/{repo_path}?ref={BRANCH}"
    )
    if status == 200 and isinstance(response, dict):
        return response.get("sha")
    return None


def commit_file(token, local_path, repo_path):
    """
    Коммитит один файл - создаёт или обновляет.
    Если это .ipynb - чистим metadata.widgets, иначе GitHub
    выдаёт "Invalid Notebook" из-за отсутствующего ключа state.
    """
    # для ноутбуков делаем очистку метаданных перед коммитом
    if local_path.endswith(".ipynb"):
        with open(local_path, "r", encoding="utf-8") as f:
            nb_data = json.load(f)
        # этот блок виджетов добавляет ipywidgets, но без ключа state -
        # GitHub-renderer на это ругается. Просто удалим его.
        if "widgets" in nb_data.get("metadata", {}):
            del nb_data["metadata"]["widgets"]
        # также почистим от viewable widgets-output, чтобы файл рендерился чисто
        for cell in nb_data.get("cells", []):
            for out in cell.get("outputs", []):
                data = out.get("data", {})
                if "application/vnd.jupyter.widget-view+json" in data:
                    # оставляем text/plain если есть, иначе убираем сам output
                    pass
        content_bytes = json.dumps(nb_data, ensure_ascii=False, indent=1).encode("utf-8")
    else:
        with open(local_path, "rb") as f:
            content_bytes = f.read()

    content_b64 = base64.b64encode(content_bytes).decode("utf-8")

    existing_sha = get_existing_sha(token, repo_path)

    payload = {
        "message": COMMIT_MESSAGE,
        "content": content_b64,
        "branch": BRANCH,
    }
    if existing_sha is not None:
        payload["sha"] = existing_sha

    status, response = github_api(
        token, "PUT", f"/repos/{OWNER}/{REPO}/contents/{repo_path}", payload
    )

    if status in (200, 201):
        action = "обновлён" if existing_sha else "создан"
        commitSha = ""
        if isinstance(response, dict):
            commitSha = response.get("commit", {}).get("sha", "")[:7]
        print(f"  OK   {repo_path}  ({action}, commit {commitSha})")
        return True

    # вытаскиваем понятное сообщение от GitHub
    if isinstance(response, dict):
        errMsg = response.get("message", str(response))
        if status == 403:
            errMsg = errMsg + "  (нет прав - токен должен иметь Contents: Read and write на этот репо)"
        if status == 401:
            errMsg = errMsg + "  (токен невалиден или истёк - создай новый)"
        if status == 404:
            errMsg = errMsg + "  (репо не найдено или токен не видит этот репо в Repository access)"
    else:
        errMsg = str(response)[:300]
    print(f"  FAIL {repo_path}  status={status}")
    print(f"       {errMsg}")
    return False


def get_token(secret_name):
    """
    Получает токен. Источники по порядку:
      1) секреты Colab (google.colab.userdata)
      2) переменная окружения с тем же именем
      3) поле ручного ввода (Password)
    Возвращает (token, source).
    """
    # 1) секреты Colab
    try:
        from google.colab import userdata
        try:
            value = userdata.get(secret_name)
            if value:
                return value, f"секрет Colab '{secret_name}'"
        except Exception:
            pass
    except ImportError:
        pass

    # 2) переменная окружения
    envValue = os.environ.get(secret_name, "").strip()
    if envValue:
        return envValue, f"переменная окружения {secret_name}"

    # 3) ручной ввод
    manual = manual_token_input.value.strip()
    if manual:
        return manual, "поле ручного ввода"

    return "", ""


def check_token_scopes(token):
    """
    Делает диагностический запрос - проверяет что токен валиден
    и какие у него scopes/permissions. Помогает быстро понять,
    почему PUT падает с 403.
    """
    status, response = github_api(token, "GET", f"/repos/{OWNER}/{REPO}")
    if status == 200:
        permissions = response.get("permissions", {}) if isinstance(response, dict) else {}
        canPush = permissions.get("push", False)
        canAdmin = permissions.get("admin", False)
        print(f"  Доступ к репо: push={canPush}, admin={canAdmin}")
        if not canPush:
            print("  ВНИМАНИЕ: у токена нет права push (write) - PUT упадёт с 403.")
            print("  Создай новый токен с Contents: Read and write на этот репо.")
        return canPush
    elif status == 404:
        print(f"  Не вижу репо {OWNER}/{REPO} - токен не имеет доступа к нему.")
        print("  Проверь Repository access в настройках токена.")
        return False
    else:
        msg = response.get("message", "") if isinstance(response, dict) else str(response)
        print(f"  Проверка репо вернула status={status}: {msg}")
        return False


def on_commit_click(_):
    """
    Обработчик нажатия кнопки - получаем токен, проверяем, коммитим.
    """
    with output_area:
        clear_output()
        secretName = secret_name_input.value.strip() or "GH_TOKEN"
        token, source = get_token(secretName)

        if not token:
            print(f"Не нашёл токен в секрете '{secretName}'.")
            print("Проверь что:")
            print(f"  - в панели 'Секреты' Colab есть секрет с именем {secretName}")
            print("  - переключатель 'Доступ из блокнотов' включён")
            print("  - либо введи токен в поле ручного ввода и нажми кнопку ещё раз")
            return

        print(f"Токен взят из: {source}")
        print(f"Целевой репо:  {OWNER}/{REPO}, ветка {BRANCH}")
        print()

        # быстро проверим права токена ДО попытки коммита
        print("Проверяю права токена...")
        ok = check_token_scopes(token)
        print()
        if not ok:
            print("Прерываю - сначала исправь права токена.")
            return

        # собираем итоговый список файлов: ноутбук (если найден) + результаты
        filesToCommit = dict(DATA_FILES)
        nbPath = find_notebook_path()
        if nbPath:
            filesToCommit[nbPath] = "final/final_hw_efrsb.ipynb"
            print(f"Ноутбук найден здесь: {nbPath}")
        else:
            print("Ноутбук не найден локально - закоммичу только файлы-результаты.")
            print("(в Colab сначала Файл -> Сохранить копию в Drive, чтобы файл был на диске)")

        print()
        print("Начинаю коммит...")
        successCount = 0
        failCount = 0
        for localPath, repoPath in filesToCommit.items():
            if not os.path.exists(localPath):
                print(f"  SKIP {repoPath}  (нет локального файла {localPath})")
                failCount = failCount + 1
                continue
            ok = commit_file(token, localPath, repoPath)
            if ok:
                successCount = successCount + 1
            else:
                failCount = failCount + 1

        print()
        print(f"Готово. Успешно: {successCount}, ошибок: {failCount}")
        if successCount > 0:
            print(f"Проверь репозиторий: https://github.com/{OWNER}/{REPO}/tree/{BRANCH}/final")


# создаём виджеты
secret_name_input = widgets.Text(
    value="GH_TOKEN",
    description="Имя секрета:",
    layout=widgets.Layout(width="400px"),
)
manual_token_input = widgets.Password(
    description="или токен:",
    placeholder="github_pat_... (если не используешь секрет)",
    layout=widgets.Layout(width="500px"),
)
btn_commit = widgets.Button(
    description="Закоммитить в GitHub",
    button_style="warning",
    icon="github",
    layout=widgets.Layout(width="220px", height="40px"),
)
output_area = widgets.Output()

btn_commit.on_click(on_commit_click)

display(secret_name_input)
display(manual_token_input)
display(btn_commit)
display(output_area)

print("Сначала проверим права токена, потом закоммитим файлы.")
print("Если будет 403 - в выводе будет точное сообщение GitHub и подсказка что исправить.")


Text(value='GH_TOKEN', description='Имя секрета:', layout=Layout(width='400px'))

Password(description='или токен:', layout=Layout(width='500px'), placeholder='github_pat_... (если не использу…

Button(button_style='warning', description='Закоммитить в GitHub', icon='github', layout=Layout(height='40px',…

Output()

Сначала проверим права токена, потом закоммитим файлы.
Если будет 403 - в выводе будет точное сообщение GitHub и подсказка что исправить.
